Assume the latent factor model 
$$
\mathbf{Y}_t = \boldsymbol{\Theta}_t\mathbf{Z} + \boldsymbol{\Lambda}_t \mathbf{M},
$$ 
where $t\in 1,\ldots, T$ and the matrices $\boldsymbol{\Theta}_t$ and $\boldsymbol{\Lambda}_t$ are:
$$
\begin{pmatrix}
\theta_{t_{11}}&\dots &\theta_{t_{1p}} \\
\vdots & & \vdots \\
\theta_{t_{m1}} &\dots&\theta_{t_{mp}}
\end{pmatrix} \quad\text{and}\quad
\begin{pmatrix}
\lambda_{t_{11}}&\dots &\lambda_{t_{1f}} \\
\vdots & & \vdots \\
\lambda_{t_{m1}} &\dots&\lambda_{t_{mf}}
\end{pmatrix}
$$ 
and the matrices $\mathbf{Z}$ and $\mathbf{M}$ are
$$
\begin{pmatrix}
Z_{11}&\dots & Z_{N-1,1} \\
\vdots & & \vdots \\
Z_{1p} &\dots& Z_{N-1, p}
\end{pmatrix} \quad\text{and}\quad
\begin{pmatrix}
\mu_{11}&\dots & \mu_{N-1,1} \\
\vdots & & \vdots \\
\mu_{1p} &\dots& \mu_{N-1, p}
\end{pmatrix}
$$ 

In [140]:
import numpy as np
import torch
import learnQ as lq

torch.manual_seed(42)
np.random.seed(42)

def sim_Data(T, N, m, p, f, with_covariates):
    Outcomes = []
    Thetas = []
    Lambdas = []
    latent_terms = []

    # Fixed base loadings
    Theta_base = torch.randn(m, p, dtype=torch.float64)
    Lambda_base = torch.randn(m, f, dtype=torch.float64)

    # step directions
    Theta_step = torch.randn(m, p, dtype=torch.float64)
    Lambda_step = torch.randn(m, f, dtype=torch.float64)

    for t in range(T):
        if t == 0:
            # start out at the base, before taking any steps
            Theta_t = Theta_base
            Lambda_t = Lambda_base
        else:
            # take a step, always in the same direction, but by varying amounts
            Theta_t = Theta_base + (Theta_step * torch.abs(torch.randn(1)))
            Lambda_t = Lambda_base + (Lambda_step * torch.abs(torch.randn(1)))
        # each time point append
        Thetas.append(Theta_t)
        Lambdas.append(Lambda_t)

    # if we aren't using covariates, just do the loadings for the latent factors
    Z_donors = torch.randn(p, N, dtype=torch.float64) * bool(with_covariates)  # (p x N)
    mu_donors = torch.randn(f, N, dtype=torch.float64)  # (f x N)

    

    for t in range(T):
        # returning the latent terms
        L_t = Lambdas[t] @ mu_donors
        latent_terms.append(L_t)

        Y_t = Thetas[t] @ Z_donors + Lambdas[t] @ mu_donors
        Outcomes.append(Y_t)

    return Outcomes, latent_terms






In [ ]:
torch.manual_seed(42)

# settings
T = 20
N = 10
m = 3
p = 4
f = 5
with_covariates = False

# run the DGP without covariates
Outcomes, latent_terms = sim_Data(T, N, m, p, f, with_covariates)

# prepare targets and covariates
targets = []
covariates = []
for outcome in Outcomes:
    # take the first column
    targets.append(outcome[:, 0])
    # all but the first column
    covariates.append(outcome[:, 1:])

# run the learning algorithm
Q_final, w_final = lq.learnQ(targets, covariates, embedding_dim=5, n_iterations=2000, reg_Q=0, reg_w=0, verbose=False)


In [154]:
Q_final
w_final

Q_final @ w_final
print(Q_final)


[[ 0.10866108 -0.39501788 -1.774782    0.46118823 -0.41898701]
 [-0.16219756  1.24964847 -0.70843959 -0.46172278 -1.21734373]
 [-0.38555082  0.5099628  -0.08234379 -0.9333187  -1.65637269]
 [-1.15316979  1.33670808  0.74231002  1.6488994  -0.6416569 ]
 [-0.6973585   0.20549526 -0.01003066  0.63064201 -1.3081931 ]
 [-0.0486342   0.99940554  0.97135651  0.65833488  0.65189518]
 [-0.16428284  0.94516142 -1.60888732  0.95828784  0.05237856]
 [ 0.65930967 -2.05684341  0.63670014 -0.68968727 -0.33208128]
 [ 0.67425132  0.49077584 -0.8604157   2.47369937 -1.45094827]]


In [ ]:
# The final loss can be large while this mean is small. What does that indicate?
# try passing the covariates also. Feels like given this info it should be able to sus out the latent factors.


diff = [((A.numpy().T @ A.numpy() @ (Q_final @ w_final)) - (A.numpy().T @ d.numpy())) for A, d in zip(covariates, targets)]
print(diff)
print(np.mean(diff))